# Hybrid LSTM-Transformer: Nifty 50 Next-Day Close Prediction
Single-file project: data, model, MC dropout, backtest and the Streamlit app all live here.
Runtime: **GPU** (Runtime > Change runtime type > T4). Run cells top to bottom; the last cell starts the app and prints its link.

Design note: the network predicts the **next-day log return (in %)**, and the price is reconstructed as `Close_T * exp(r/100)`. Predicting raw price levels fails on the 2023+ test set because prices go beyond the training range. The output is still an exact next-day close.

In [ ]:
!pip -q install yfinance kagglehub openpyxl

In [ ]:
import os, json, math, random, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from numpy.lib.stride_tricks import sliding_window_view
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ---------------- Config ----------------
SEQ_LEN   = 60
TRAIN_END = "2017-12-31"
VAL_END   = "2022-12-31"
MC_PASSES = 100
EPOCHS    = 60
BATCH     = 64
LR        = 1e-4
PATIENCE  = 12
COST      = 0.0002   # per side transaction cost in backtest

## Step 1: Data collection, merging & cleaning
All three Kaggle datasets from the project plan are downloaded with `kagglehub` (no upload needed) and merged in **priority order**: a date is taken from the first source that has it, later sources only fill gaps. Yahoo Finance tops up the latest days.

Each dataset is scanned file by file and only **Nifty 50 index data** is accepted. Per-stock files (for example in `rohanrao/...`) are skipped, because mixing stock prices into index prices would corrupt the series. The printout shows what every source contributed. If Kaggle asks for credentials, add `KAGGLE_USERNAME` / `KAGGLE_KEY` in Colab Secrets (key icon, left sidebar).

In [ ]:
import re, glob
import kagglehub, yfinance as yf

# priority order: earlier wins, later only fills missing dates
KAGGLE_SOURCES = [
    ("aparnamadathil/nifty-50-1999-2026-full-27-years-data", "daily index, 1999-2026"),
    ("rohanrao/nifty50-stock-market-data",                  "2000-2021 (used only if it has index rows)"),
    ("debashis74017/nifty-50-minute-data",                  "1-minute -> resampled to daily"),
]
USE_YAHOO = True      # fills the most recent days / gaps (lowest priority)

ALIASES = {
    "date":   ["date", "datetime", "timestamp", "time"],
    "open":   ["open", "openprice"],
    "high":   ["high", "highprice"],
    "low":    ["low", "lowprice"],
    "close":  ["close", "closeprice", "price", "last"],
    "volume": ["volume", "vol", "sharestraded"],
}
SYMBOL_COLS = ["symbol", "ticker", "stock", "company", "name", "instrument"]
INDEX_NAMES = {"NIFTY", "NIFTY50", "CNXNIFTY"}

def parse_volume(s):
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors="coerce")
    mult = {"K": 1e3, "M": 1e6, "B": 1e9}
    def f(x):
        x = str(x).replace(",", "").strip().upper()
        if x in ("", "-", "NAN"): return np.nan
        try:
            return float(x[:-1]) * mult[x[-1]] if x[-1] in mult else float(x)
        except ValueError:
            return np.nan
    return s.map(f)

def parse_dates(s):
    s = s.astype(str)
    has_tz = s.str.contains(r"\d{2}:\d{2}(?::\d{2})?(?:\.\d+)?(?:[+-]\d{2}:?\d{2}|Z)$", regex=True).mean() > 0.5
    if has_tz:
        return pd.to_datetime(s, errors="coerce", utc=True).dt.tz_convert("Asia/Kolkata").dt.tz_localize(None)
    t = pd.to_datetime(s, errors="coerce")
    if t.isna().mean() > 0.02:                         # maybe DD-MM-YYYY
        t2 = pd.to_datetime(s, errors="coerce", dayfirst=True)
        if t2.isna().mean() < t.isna().mean():
            t = t2
    return t

def normalize(raw):
    key = {c: re.sub(r"[^a-z]", "", str(c).lower()) for c in raw.columns}
    out = {}
    for std, names in ALIASES.items():
        for col, k in key.items():
            if k in names and std not in out:
                out[std] = raw[col]
    if not {"date", "open", "high", "low", "close"} <= set(out):
        raise ValueError(f"cannot map columns {list(raw.columns)}")
    d = pd.DataFrame(out)
    d["date"] = parse_dates(d["date"])
    for c in ["open", "high", "low", "close"]:
        d[c] = pd.to_numeric(d[c].astype(str).str.replace(",", ""), errors="coerce")
    d["volume"] = parse_volume(d["volume"]) if "volume" in d else 0.0
    d = d.dropna(subset=["date", "close"]).set_index("date").sort_index()
    if d.index.normalize().duplicated().mean() > 0.5:      # intraday -> daily bars
        d = d.groupby(d.index.normalize()).agg(open=("open", "first"), high=("high", "max"),
                                               low=("low", "min"), close=("close", "last"),
                                               volume=("volume", "sum"))
    d.index = d.index.normalize()
    d = d[~d.index.duplicated(keep="last")]
    d.columns = [c.capitalize() for c in d.columns]
    return d

BAD_WORDS = r"bank|next|mid|small|fin|pharma|auto|fmcg|metal|energy|realty|media|psu|infra|nifty(?:100|200|500)"
def is_index_name(path):
    n = re.sub(r"[^a-z0-9]", "", os.path.splitext(os.path.basename(path))[0].lower())
    return bool(re.search(r"nifty50", n)) and not re.search(BAD_WORDS, n)

def index_rows_only(raw):
    """Return raw rows that belong to the Nifty 50 INDEX, or None if the file is stock-level data."""
    sym = next((c for c in raw.columns if re.sub(r"[^a-z]", "", str(c).lower()) in SYMBOL_COLS), None)
    if sym is None:
        return raw                                          # index-only file (no symbol column)
    vals = raw[sym].astype(str).str.upper().str.replace(r"[^A-Z0-9]", "", regex=True)
    hit = vals.isin(INDEX_NAMES)
    return raw[hit] if hit.any() else None                  # stock files have no index rows

def load_kaggle(slug, note):
    print(f"\n=== {slug}  ({note})")
    try:
        path = kagglehub.dataset_download(slug)
    except Exception as e:
        print(f"  download failed: {type(e).__name__}: {str(e)[:150]}")
        print("  -> add KAGGLE_USERNAME / KAGGLE_KEY to Colab Secrets, or call kagglehub.login()")
        return None
    files = [f for f in glob.glob(f"{path}/**/*", recursive=True)
             if f.lower().endswith((".csv", ".xlsx", ".xls"))
             and not any(w in os.path.basename(f).lower() for w in ("metadata",))]
    print(f"  {len(files)} data files found")
    matched = [f for f in files if is_index_name(f)]      # pick by filename first: do NOT read hundreds of stock files
    if matched:
        files = matched
        print("  candidate index files:", [os.path.basename(f) for f in files][:5])
    elif len(files) > 60:
        print("  no file named like Nifty 50 among many files -> skipped (not reading stock-level files)")
        return None
    best, skipped = None, 0
    for f in files:
        try:
            raw = pd.read_excel(f) if f.lower().endswith((".xlsx", ".xls")) else pd.read_csv(f)
            raw = index_rows_only(raw)
            if raw is None:
                skipped += 1; continue
            d = normalize(raw)
        except Exception:
            skipped += 1; continue
        if len(d) > 100 and (best is None or len(d) > len(best[1])):
            best = (os.path.basename(f), d)
    if best is None:
        print(f"  no usable Nifty 50 INDEX data (skipped {skipped} files: stock-level or unreadable)")
        return None
    name, d = best
    print(f"  using {name}: {len(d)} daily rows, {d.index.min().date()} to {d.index.max().date()}")
    return d

frames, names = [], []
for slug, note in KAGGLE_SOURCES:
    d = load_kaggle(slug, note)
    if d is not None:
        frames.append(d); names.append(slug.split("/")[0])

if USE_YAHOO:
    y = yf.download("^NSEI", start="2007-09-01", auto_adjust=False, progress=False)
    if isinstance(y.columns, pd.MultiIndex):
        y.columns = y.columns.get_level_values(0)
    y = y[["Open", "High", "Low", "Close", "Volume"]]
    y.index = pd.to_datetime(y.index).tz_localize(None).normalize()
    frames.append(y); names.append("yahoo")
    print(f"\n=== Yahoo ^NSEI: {len(y)} rows, {y.index.min().date()} to {y.index.max().date()}")

assert frames, "No data source worked. Check Kaggle credentials / internet."

# cross-check: how well do overlapping sources agree on Close?
print("\n--- agreement with first source ---")
for n, f_ in zip(names[1:], frames[1:]):
    ov = frames[0].index.intersection(f_.index)
    if len(ov):
        diff = (frames[0].loc[ov, "Close"] / f_.loc[ov, "Close"] - 1).abs()
        print(f"{names[0]} vs {n}: {len(ov)} common days, median |diff| {diff.median():.4%}, max {diff.max():.2%}")

df = frames[0]
for f_ in frames[1:]:
    df = df.combine_first(f_)          # earlier sources win; later only fill gaps
df = df[["Open", "High", "Low", "Close", "Volume"]].sort_index()

# how many days came from each source
src = pd.Series("", index=df.index)
for n, f_ in reversed(list(zip(names, frames))):
    src[src.index.isin(f_.index)] = n
print("\n--- rows contributed per source (after priority merge) ---")
print(src.value_counts().to_string())

price_cols = ["Open", "High", "Low", "Close"]
df[price_cols] = df[price_cols].replace(0, np.nan)
df[price_cols] = df[price_cols].ffill(limit=3).interpolate(limit_direction="both")  # short gaps ffill, longer interpolate
df["Volume"] = df["Volume"].fillna(0)
df = df.dropna(subset=["Close"])
df["log_return"] = np.log(df["Close"] / df["Close"].shift(1))
print("\nFINAL:", df.shape, df.index.min().date(), "->", df.index.max().date())
print("Daily moves >12% (check for bad rows):", int((df["log_return"].abs() > 0.12).sum()))
df.tail()

## Step 2: Feature engineering (rolling z-score, window = 252, past-only)
Written to `features.py` so the Streamlit app uses the exact same code.

In [ ]:
%%writefile features.py
import numpy as np
import pandas as pd


def build_features(df):
    """df columns: Open, High, Low, Close, Volume (daily, sorted)."""
    c, h, l, v = df.Close, df.High, df.Low, df.Volume
    f = pd.DataFrame(index=df.index)
    f["log_ret"]   = np.log(c / c.shift(1))
    # trend
    f["sma20"]     = c.rolling(20).mean()
    f["sma50"]     = c.rolling(50).mean()
    f["ema20"]     = c.ewm(span=20, adjust=False).mean()
    f["price_sma"] = c / f["sma20"]
    # momentum
    d = c.diff()
    gain = d.clip(lower=0).ewm(alpha=1/14, adjust=False).mean()
    loss = (-d.clip(upper=0)).ewm(alpha=1/14, adjust=False).mean()
    f["rsi14"]     = 100 - 100 / (1 + gain / (loss + 1e-12))
    macd = c.ewm(span=12, adjust=False).mean() - c.ewm(span=26, adjust=False).mean()
    sig  = macd.ewm(span=9, adjust=False).mean()
    f["macd"], f["macd_sig"], f["macd_hist"] = macd, sig, macd - sig
    # volatility
    f["std20"]     = f["log_ret"].rolling(20).std()
    f["bb_width"]  = 4 * c.rolling(20).std() / f["sma20"]
    tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    f["atr14"]     = tr.ewm(alpha=1/14, adjust=False).mean()
    # volume (index volume is 0 in some years -> safe forms)
    f["vol_sma20"] = np.log1p(v).rolling(20).mean()
    f["vol_ratio"] = v / (v.rolling(20).mean() + 1)
    # intraday range
    f["hl_range"]  = (h - l) / c
    return f


def make_z(raw, window=252):
    """Rolling z-score using only past/current data, clipped to +-5."""
    roll = raw.rolling(window)
    return ((raw - roll.mean()) / (roll.std() + 1e-8)).clip(-5, 5).dropna()

In [ ]:
import importlib, features
importlib.reload(features)
from features import build_features, make_z

raw = build_features(df)
Z = make_z(raw)
FEATS = list(Z.columns)
print(len(FEATS), "features:", FEATS)

## Step 3: Sequence construction + chronological split
Target = next-day log return (%). Row `t` uses only data up to close of day `t`.

In [ ]:
close  = df["Close"].loc[Z.index]
tgt    = (np.log(df["Close"].shift(-1) / df["Close"]) * 100).loc[Z.index]   # %
open_n = df["Open"].shift(-1).loc[Z.index]
close_n= df["Close"].shift(-1).loc[Z.index]

F = Z.values.astype(np.float32)
N = len(F)
W = sliding_window_view(F, SEQ_LEN, axis=0).transpose(0, 2, 1)   # (N-SEQ+1, SEQ, d)
idx   = np.arange(SEQ_LEN - 1, N)
dates = Z.index[idx]
y_all = tgt.values[idx]
valid = ~np.isnan(y_all)

latest_seq = np.ascontiguousarray(W[-1:])           # for live prediction (target unknown)
latest_date, latest_close = dates[-1], close.values[idx][-1]

X       = np.ascontiguousarray(W[valid])
y       = y_all[valid].astype(np.float32)
d_      = dates[valid]
c_t     = close.values[idx][valid]
o_n     = open_n.values[idx][valid]
c_n     = close_n.values[idx][valid]

m_tr = d_ <= pd.Timestamp(TRAIN_END)
m_va = (d_ > pd.Timestamp(TRAIN_END)) & (d_ <= pd.Timestamp(VAL_END))
m_te = d_ > pd.Timestamp(VAL_END)
print("train/val/test:", m_tr.sum(), m_va.sum(), m_te.sum())

# keep target scale readable for Huber (std of daily returns is ~1%)
loaders = {
    "train": DataLoader(TensorDataset(torch.tensor(X[m_tr]), torch.tensor(y[m_tr])), batch_size=BATCH, shuffle=True),
    "val":   DataLoader(TensorDataset(torch.tensor(X[m_va]), torch.tensor(y[m_va])), batch_size=256),
}

## Step 4: Model (written to model.py so Streamlit can import it)

In [ ]:
%%writefile model.py
import math
import torch
import torch.nn as nn


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]


class HybridLSTMTransformer(nn.Module):
    """LSTM encoder -> Transformer encoder -> GAP -> Dense(64) -> Dropout -> Dense(1).
    Output = predicted next-day log return in % (price = close * exp(out/100))."""

    def __init__(self, input_dim=15, hidden_dim=128, lstm_layers=2, n_heads=4,
                 tf_layers=2, ff_dim=256, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=lstm_layers,
                            batch_first=True, dropout=dropout)
        self.pos = PositionalEncoding(hidden_dim)
        layer = nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=n_heads,
                                           dim_feedforward=ff_dim, dropout=0.1,
                                           batch_first=True)
        self.tf = nn.TransformerEncoder(layer, num_layers=tf_layers)
        self.head = nn.Sequential(nn.Linear(hidden_dim, 64), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(64, 1))

    def forward(self, x):
        h, _ = self.lstm(x)
        h = self.tf(self.pos(h))
        return self.head(h.mean(dim=1)).squeeze(-1)

In [ ]:
from model import HybridLSTMTransformer

model = HybridLSTMTransformer(input_dim=len(FEATS), hidden_dim=128).to(device)
print(sum(p.numel() for p in model.parameters()), "parameters")

criterion = nn.HuberLoss(delta=1.0)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=5)

## Step 5a: Training (early stopping on validation loss)

In [ ]:
def run_epoch(loader, train):
    model.train(train)
    tot, n = 0.0, 0
    with torch.set_grad_enabled(train):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = criterion(model(xb), yb)
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            tot += loss.item() * len(xb); n += len(xb)
    return tot / n

best, bad, hist = 1e9, 0, []
for ep in range(1, EPOCHS + 1):
    tr = run_epoch(loaders["train"], True)
    va = run_epoch(loaders["val"], False)
    scheduler.step(va)
    hist.append((tr, va))
    if va < best - 1e-5:
        best, bad = va, 0
        torch.save(model.state_dict(), "best.pt")
    else:
        bad += 1
    print(f"ep {ep:02d} | train {tr:.4f} | val {va:.4f} | lr {optimizer.param_groups[0]['lr']:.1e}")
    if bad >= PATIENCE:
        print("Early stop"); break

model.load_state_dict(torch.load("best.pt"))
plt.plot([h[0] for h in hist], label="train"); plt.plot([h[1] for h in hist], label="val")
plt.legend(); plt.title("Huber loss"); plt.show()

## Step 5b: Monte Carlo dropout inference (100 passes, dropout ON)

In [ ]:
def mc_predict(model, X, n=MC_PASSES):
    model.train()                       # keeps dropout active (no BatchNorm in model, so safe)
    Xt = torch.tensor(X)
    out = np.zeros((n, len(X)), dtype=np.float32)
    with torch.no_grad():
        for i in range(n):
            out[i] = torch.cat([model(Xt[j:j+512].to(device)).cpu()
                                for j in range(0, len(Xt), 512)]).numpy()
    return out

def to_price(ret_pct, c_t):
    return c_t[None, :] * np.exp(ret_pct / 100.0)

def summarize(passes_pct, c_t):
    P = to_price(passes_pct, c_t)
    return P.mean(0), np.percentile(P, 5, axis=0), np.percentile(P, 95, axis=0)

res = {}
for name, m in [("val", m_va), ("test", m_te)]:
    mean, lo, hi = summarize(mc_predict(model, X[m]), c_t[m])
    res[name] = dict(mean=mean, lo=lo, hi=hi, actual=c_n[m], c_t=c_t[m], open_n=o_n[m], dates=d_[m])

# Interval calibration: MC dropout alone is usually too narrow -> scale width on VALIDATION so 90% coverage holds
v = res["val"]
half = np.maximum((v["hi"] - v["lo"]) / 2, 1e-6)
K = float(np.quantile(np.abs(v["actual"] - v["mean"]) / half, 0.90))
print("Interval scale factor K (fit on validation):", round(K, 2))
for r in res.values():
    r["lo_cal"] = r["mean"] - K * (r["mean"] - r["lo"])
    r["hi_cal"] = r["mean"] + K * (r["hi"] - r["mean"])

## Step 6: Evaluation on test set (with naive "tomorrow = today" baseline)

In [ ]:
def metrics(actual, pred):
    err = actual - pred
    return dict(MAE=np.mean(np.abs(err)), RMSE=np.sqrt(np.mean(err**2)),
                MAPE=np.mean(np.abs(err / actual)) * 100,
                R2=1 - np.sum(err**2) / np.sum((actual - actual.mean())**2))

t = res["test"]
rows = {"Hybrid LSTM-Transformer": metrics(t["actual"], t["mean"]),
        "Naive (close_T)":          metrics(t["actual"], t["c_t"])}
print(pd.DataFrame(rows).T.round(3))
print("Reference LSTM from GitHub project: MAE 116.82 | RMSE 167.68")

dir_acc = np.mean(np.sign(t["mean"] - t["c_t"]) == np.sign(t["actual"] - t["c_t"]))
cov_raw = np.mean((t["actual"] >= t["lo"]) & (t["actual"] <= t["hi"]))
cov_cal = np.mean((t["actual"] >= t["lo_cal"]) & (t["actual"] <= t["hi_cal"]))
print(f"Direction accuracy: {dir_acc:.1%}")
print(f"90% interval coverage  raw MC: {cov_raw:.1%} | calibrated: {cov_cal:.1%}  (target 85-95%)")

n = 120
plt.figure(figsize=(12, 4))
plt.plot(t["dates"][-n:], t["actual"][-n:], label="Actual close")
plt.plot(t["dates"][-n:], t["mean"][-n:], label="Predicted close")
plt.fill_between(t["dates"][-n:], t["lo_cal"][-n:], t["hi_cal"][-n:], alpha=0.25, label="90% interval (calibrated)")
plt.legend(); plt.title("Test set: last 120 days"); plt.show()

## Step 7: Backtest (entry at T+1 open, exit at T+1 close) vs buy-and-hold
Threshold is tuned on the **validation** set, then applied unchanged to test.

In [ ]:
def backtest(r, thr, cost=COST):
    up = r["mean"] > r["c_t"] * (1 + thr)
    dn = r["mean"] < r["c_t"] * (1 - thr)
    sig = np.where(up, 1, np.where(dn, -1, 0))
    intraday = r["actual"] / r["open_n"] - 1
    ret = sig * intraday - np.abs(sig) * cost * 2
    return sig, ret

def stats(ret):
    eq = np.cumprod(1 + ret)
    yrs = len(ret) / 252
    sharpe = ret.mean() / (ret.std() + 1e-12) * np.sqrt(252)
    mdd = (eq / np.maximum.accumulate(eq) - 1).min()
    return dict(CAGR=eq[-1] ** (1 / yrs) - 1, Sharpe=sharpe, MaxDD=mdd, FinalEquity=eq[-1]), eq

grid = [0.0, 0.0005, 0.001, 0.002, 0.003, 0.005]
val_sharpe = {g: stats(backtest(res["val"], g)[1])[0]["Sharpe"] for g in grid}
THR = max(val_sharpe, key=val_sharpe.get)
print("Validation Sharpe by threshold:", {k: round(v, 2) for k, v in val_sharpe.items()})
print("Chosen threshold:", THR)

sig, ret = backtest(t, THR)
print("Signals (SELL/HOLD/BUY):", (sig == -1).sum(), (sig == 0).sum(), (sig == 1).sum())
s_strat, eq_strat = stats(ret)
bh_ret = t["actual"] / t["c_t"] - 1
s_bh, eq_bh = stats(bh_ret)
print(pd.DataFrame({"Strategy": s_strat, "Buy&Hold": s_bh}).round(3))

plt.figure(figsize=(12, 4))
plt.plot(t["dates"], eq_strat, label="Strategy"); plt.plot(t["dates"], eq_bh, label="Buy & Hold")
plt.legend(); plt.title("Equity curve (test)"); plt.show()

## Live prediction for the next trading day
Run after market close (data for the latest day must be final).

In [ ]:
passes = mc_predict(model, latest_seq)                 # (100, 1) in % return
P = to_price(passes, np.array([latest_close]))[:, 0]
pt = P.mean()
lo = pt - K * (pt - np.percentile(P, 5))
hi = pt + K * (np.percentile(P, 95) - pt)
chg = pt / latest_close - 1
signal = "BUY" if chg > THR else ("SELL" if chg < -THR else "HOLD")

print(f"Latest data date : {latest_date.date()}  (close {latest_close:,.2f})")
print(f"Predicted close  : {pt:,.2f}  ({chg:+.2%})")
print(f"90% interval     : [{lo:,.0f} - {hi:,.0f}]")
print(f"Signal           : {signal}")

## Export artifacts for the Streamlit app
Weights, `meta.json` and `backtest.csv` are saved in `models/` for the app in the next step.

In [ ]:
os.makedirs("models", exist_ok=True)
torch.save(model.state_dict(), "models/hybrid_lstm_transformer.pt")
meta = dict(features=FEATS, seq_len=SEQ_LEN, input_dim=len(FEATS), hidden_dim=128,
            target="next_day_log_return_pct", interval_scale_K=K, signal_threshold=THR,
            mc_passes=MC_PASSES, test_MAE=float(rows["Hybrid LSTM-Transformer"]["MAE"]),
            test_RMSE=float(rows["Hybrid LSTM-Transformer"]["RMSE"]),
            test_MAPE=float(rows["Hybrid LSTM-Transformer"]["MAPE"]),
            test_R2=float(rows["Hybrid LSTM-Transformer"]["R2"]))
meta["trained_on"] = f"{d_.min().date()} to {d_.max().date()}"
meta["test_period"] = f"{t['dates'].min().date()} to {t['dates'].max().date()}"
meta["model_version"] = "v1-" + pd.Timestamp.today().strftime("%Y%m%d")
json.dump(meta, open("models/meta.json", "w"), indent=2)
pd.DataFrame({"date": t["dates"], "actual": t["actual"], "pred": t["mean"], "lo": t["lo_cal"],
              "hi": t["hi_cal"], "close_t": t["c_t"], "eq_strategy": eq_strat, "eq_buyhold": eq_bh}
             ).to_csv("models/backtest.csv", index=False)
DOWNLOAD_BUNDLE = False   # set True to download model.py/features.py/models/ for deploying elsewhere
if DOWNLOAD_BUNDLE:
    !zip -qr nifty_model_bundle.zip model.py features.py models
    from google.colab import files
    files.download("nifty_model_bundle.zip")
print("Saved: models/hybrid_lstm_transformer.pt, models/meta.json, models/backtest.csv")

## Step 8: Streamlit app (runs inside this notebook)
The app file is written below and served from Colab through a free Cloudflare tunnel.

In [ ]:
%%writefile app.py
import json
import os
from datetime import datetime, time
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import streamlit as st
import torch
import yfinance as yf

from features import build_features, make_z
from model import HybridLSTMTransformer

IST = ZoneInfo("Asia/Kolkata")
MARKET_CLOSE = time(15, 30)
DATA_FINAL = time(16, 0)
MC_PASSES = 100

st.set_page_config(page_title="Nifty 50 Next-Day Close Prediction", page_icon="📈", layout="wide")


@st.cache_resource
def load_model():
    with open("models/meta.json") as f:
        meta = json.load(f)
    m = HybridLSTMTransformer(input_dim=meta["input_dim"], hidden_dim=meta["hidden_dim"])
    m.load_state_dict(torch.load("models/hybrid_lstm_transformer.pt", map_location="cpu"))
    return m, meta


@st.cache_data(ttl=1800, show_spinner="Fetching Nifty 50 data...")
def fetch_data():
    d = yf.download("^NSEI", period="4y", auto_adjust=False, progress=False)
    if isinstance(d.columns, pd.MultiIndex):
        d.columns = d.columns.get_level_values(0)
    d = d[["Open", "High", "Low", "Close", "Volume"]].dropna(subset=["Close"])
    d.index = pd.to_datetime(d.index).tz_localize(None).normalize()
    return d


@st.cache_data
def load_backtest():
    p = "models/backtest.csv"
    return pd.read_csv(p, parse_dates=["date"]) if os.path.exists(p) else None


def drop_partial_bar(d, now):
    if d.index[-1].date() == now.date() and now.time() < DATA_FINAL:
        return d.iloc[:-1]
    return d


def mc_predict(model, seq, n=MC_PASSES):
    torch.manual_seed(0)
    model.train()                                   # dropout stays ON -> MC dropout
    x = torch.tensor(seq[None], dtype=torch.float32).repeat(n, 1, 1)
    with torch.no_grad():
        return model(x).numpy()                     # next-day log return in %


model, meta = load_model()
SEQ_LEN, K = meta["seq_len"], meta["interval_scale_K"]
now = datetime.now(IST)
can_refresh = now.weekday() < 5 and now.time() >= MARKET_CLOSE

with st.sidebar:
    st.header("Controls")
    if st.button("Refresh Data", disabled=not can_refresh, use_container_width=True):
        fetch_data.clear()
        st.rerun()
    if not can_refresh:
        st.caption("Refresh unlocks after 15:30 IST on trading days. Showing the last completed session.")
    thr_pct = st.slider("Signal threshold (± % vs current close)", 0.0, 1.0,
                        float(round(meta["signal_threshold"] * 100, 2)), 0.05)
    st.caption(f"Default {meta['signal_threshold']:.2%} was tuned on the validation set.")
    st.caption(f"Now: {now.strftime('%d %b %Y, %H:%M IST')}")

st.title("Nifty 50 Next-Day Close Prediction")

try:
    data = drop_partial_bar(fetch_data(), now)
    raw = build_features(data)
    Z = make_z(raw)
except Exception as e:
    st.error(f"Could not fetch or process market data: {e}")
    st.stop()

if list(Z.columns) != meta["features"]:
    st.error("Feature mismatch between app and trained model.")
    st.stop()
if len(Z) < SEQ_LEN:
    st.error("Not enough history to build a 60-day sequence.")
    st.stop()

seq = Z.values[-SEQ_LEN:].astype(np.float32)
last_date = Z.index[-1]
closes = data["Close"].loc[:last_date]
current_close = float(closes.iloc[-1])
age_days = (now.date() - last_date.date()).days
if age_days > 4:
    st.warning(f"Latest data point is {last_date.date()} ({age_days} days old). Data feed may be stale.")

rets = mc_predict(model, seq)
prices = current_close * np.exp(rets / 100.0)
point = float(prices.mean())
lo = float(point - K * (point - float(np.percentile(prices, 5))))
hi = float(point + K * (float(np.percentile(prices, 95)) - point))
chg = float(point / current_close - 1)
thr = thr_pct / 100
signal = "BUY" if chg > thr else ("SELL" if chg < -thr else "HOLD")
next_day = last_date + pd.offsets.BDay(1)       # ignores exchange holidays

st.caption(f"Prediction generated: {now.strftime('%d %b %Y, %H:%M IST')}  ·  "
           f"based on close of {last_date.strftime('%d %b %Y')}  ·  target session: {next_day.strftime('%d %b %Y')}")

c1, c2, c3, c4 = st.columns(4)
c1.metric("Current Close", f"{current_close:,.2f}")
c2.metric("Predicted Close", f"{point:,.2f}", f"{chg:+.2%}")
c3.metric("90% Interval", f"{lo:,.0f} – {hi:,.0f}")
c4.metric("Signal", signal, f"threshold ±{thr_pct:.2f}%", delta_color="off")

hist = closes.iloc[-120:]
fig = go.Figure()
fig.add_trace(go.Scatter(x=hist.index, y=hist.values, mode="lines", name="Close"))
fig.add_trace(go.Scatter(x=[last_date, next_day, next_day, last_date], y=[current_close, hi, lo, current_close],
                         fill="toself", fillcolor="rgba(255,127,14,0.25)", line=dict(width=0), name="90% interval"))
fig.add_trace(go.Scatter(x=[next_day], y=[point], mode="markers", name="Predicted close",
                         marker=dict(size=11, color="#ff7f0e", symbol="diamond")))
fig.update_layout(title="Nifty 50: last 120 sessions + next-day forecast", height=420,
                  margin=dict(l=10, r=10, t=50, b=10), legend=dict(orientation="h"))
st.plotly_chart(fig, use_container_width=True)

bt = load_backtest()
if bt is not None:
    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(x=bt["date"], y=bt["eq_strategy"], name="Strategy"))
    fig2.add_trace(go.Scatter(x=bt["date"], y=bt["eq_buyhold"], name="Buy & Hold"))
    fig2.update_layout(title=f"Backtest equity curve (test period {meta.get('test_period', '')})",
                       height=380, margin=dict(l=10, r=10, t=50, b=10), legend=dict(orientation="h"))
    st.plotly_chart(fig2, use_container_width=True)
    st.caption("Entry at next-day open, exit at next-day close, with transaction costs; threshold tuned on validation data.")

st.divider()
st.caption(f"Model {meta.get('model_version', 'v1')} · trained on {meta.get('trained_on', 'n/a')} · "
           f"test MAE {meta['test_MAE']:.1f}, RMSE {meta['test_RMSE']:.1f}, "
           f"MAPE {meta['test_MAPE']:.2f}%, R² {meta['test_R2']:.3f}")
st.caption("Disclaimer: educational project, not investment advice. Predictions are statistical estimates and "
           "markets can move far outside the stated interval.")

In [ ]:
import subprocess, time as _t, re, sys

!pip -q install streamlit plotly
if not os.path.exists("cloudflared"):
    import urllib.request
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "cloudflared")
    os.chmod("cloudflared", 0o755)

subprocess.run("pkill -f 'streamlit run'; pkill -f cloudflared", shell=True)
subprocess.Popen([sys.executable, "-m", "streamlit", "run", "app.py", "--server.port", "8501",
                  "--server.headless", "true", "--server.enableCORS", "false",
                  "--server.enableXsrfProtection", "false"],
                 stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT, env=os.environ.copy())
_t.sleep(6)
subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
                 stdout=open("cf.log", "w"), stderr=subprocess.STDOUT)

url = None
for _ in range(40):
    _t.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("cf.log").read())
    if m:
        url = m.group(0); break
print("Open your app here:", url or "tunnel not ready, check cf.log / streamlit.log")
print("Keep this Colab runtime running while you use the app.")